# Chapter 1: Tapping Into Computational Power

This notebook accompanies **Chapter 1** of the lecture notes.

**Agenda**

🔍 · 📏 · 🗺️ · 📊 · ⚡ · 🏁

**Next steps (take it from here):** 🎯 · 🔬

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../..')
from plot_style import *
from checks import check_predict, check_mae, check_grid_search, check_operations

## 🔍 The Coffee Roasting Lab

Imagine you work in a coffee roasting lab. You have measurements of **roast temperature** (°C) and the resulting **flavor score** (1–10 scale) from a panel of tasters. Your task is to find the best linear relationship score = w x temperature + b by brute-force search over a grid of candidate parameters.

> A colleague suggests "just try every possible combination of slope and intercept." That sounds exhaustive — but how do you decide which combination is "best," and what does it cost to check them all?

<details><summary>Thought</summary>

"Best" requires a loss function that assigns a single number to each (w, b) pair — the lower the number, the better the fit. The cost depends on how finely you discretize the parameter space: halving the step size roughly quadruples the number of combinations, and each one requires evaluating the model on every data point.
</details>

In [ ]:
# Coffee roasting data: 6 roast-temperature / flavor-score pairs
roast_temp = np.array([180, 190, 200, 210, 220, 230])
flavor_score = np.array([3.2, 5.8, 7.1, 8.4, 7.9, 6.5])

print("Roast temperature (°C):", roast_temp)
print("Flavor score (1–10):   ", flavor_score)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(roast_temp, flavor_score, c=_ACCENT, s=20, linewidths=0)
ax.set_xlabel('Roast Temperature (°C)')
ax.set_ylabel('Flavor Score (1–10)')
ax.set_xlim(left=170)
ax.set_ylim(bottom=0)
tufte_axis(ax)
plt.tight_layout()
plt.show()

**Observe:**
- The six points show how flavor score changes with roast temperature.
- Scores rise with temperature, peak around 210 °C, then drop — over-roasting hurts flavor.
- A linear model won't capture the peak perfectly, but grid search will find the best straight-line approximation score = w × temperature + b.

### 🔍 Predict

> A linear model has two knobs — slope and intercept. If you set the slope correctly but forget the intercept, your line passes through the origin. Looking at the scatter plot, would a line through the origin fit the data well, or does the intercept play an important role here?

<details><summary>Thought</summary>

At 180 °C the flavor score is already 3.2. The temperatures are far from zero, so the intercept in a linear model over this range absorbs the baseline offset. Forgetting the bias would force the model to explain the high temperature values with the slope alone, leading to a poor fit.
</details>

Implement the linear model y = wx + b. Revisit the lecture notes for the formula.

Useful operations: `*` for multiplication, `+` for addition.

In [ ]:
def predict(x, w, b):
    """Return the prediction of the linear model y = w*x + b."""
    return w * x + b


check_predict(predict, 2.0, 3.8, 1.0)

### 📏 Mean Absolute Error

> Errors can be positive or negative depending on whether the model over- or under-predicts. If you simply average the raw errors, a +5 and a −5 cancel out to zero — yet neither prediction was good. How does taking the absolute value fix this?

<details><summary>Thought</summary>

The absolute value strips the sign, so every error contributes its full magnitude to the average. A prediction that is 5 units too high and one that is 5 units too low both contribute 5. The MAE for a perfect model is exactly zero; any deviation — regardless of direction — increases the score.
</details>

Implement the Mean Absolute Error between two arrays. Revisit the lecture notes for the formula.

Useful operations: `np.abs()`, `np.mean()`.

In [ ]:
def mean_absolute_error(y_true, y_pred):
    """Return the mean absolute error between y_true and y_pred."""
    return np.mean(np.abs(y_true - y_pred))


check_mae(mean_absolute_error, flavor_score, np.array([3.0, 5.5, 7.0, 8.5, 8.0, 6.8]))

### 🗺️ Grid Search

> Grid search tests every combination on a regular lattice. If the true optimum falls between two grid points, you will never find it exactly. What happens to the gap between the best grid point and the true optimum as you make the step size h smaller?

<details><summary>Thought</summary>

The maximum possible distance from any point in parameter space to the nearest grid point is proportional to h. So halving h halves the worst-case error in each parameter — the grid approximation converges to the true optimum. But the number of grid points grows as 1/h squared (for two parameters), so the computational cost rises quickly.
</details>

Implement a grid search over w and b. For each combination, compute the model predictions and the MAE. Track the best (w, b) pair and store all losses in a 2D grid for later visualization.

`w_range` and `b_range` are tuples (start, end). Use `np.arange(start, end + h, h)` to generate candidate values.

Useful operations: `np.arange()`, `np.zeros()`, `float('inf')` for initialization.

In [ ]:
def grid_search(X, Y, w_range, b_range, h):
    """Return (best_w, best_b, best_loss, loss_grid) from exhaustive grid search."""
    w_values = np.arange(w_range[0], w_range[1] + h, h)
    b_values = np.arange(b_range[0], b_range[1] + h, h)
    loss_grid = np.zeros((len(w_values), len(b_values)))

    best_w = None
    best_b = None
    best_loss = float('inf')

    for i, w in enumerate(w_values):
        for j, b in enumerate(b_values):
            preds = w * X + b
            loss = np.mean(np.abs(Y - preds))
            loss_grid[i, j] = loss
            if loss < best_loss:
                best_loss = loss
                best_w = w
                best_b = b

    return best_w, best_b, best_loss, loss_grid


check_grid_search(grid_search, roast_temp, flavor_score, (0, 0.5), (-60, 10), 0.05)

Let's run the grid search with a finer step size and inspect the result. The slope w represents how much the flavor score changes per degree, and b is the baseline offset.

In [ ]:
result = grid_search(roast_temp, flavor_score, (0, 0.5), (-60, 10), 0.01)

if result is not None:
    best_w, best_b, best_loss, loss_grid = result
    print(f"Best w = {best_w:.4f}")
    print(f"Best b = {best_b:.2f}")
    print(f"Best MAE = {best_loss:.4f}")
else:
    print("\u2b1c Grid search not implemented yet.")

### 📊 Loss Landscape

The loss grid from the previous step is a 2D array where each entry holds the MAE for one (w, b) combination. Plotting it as a heatmap reveals the shape of the loss landscape — the dark region shows where the best parameters live.

In [ ]:
if result is not None:
    w_values = np.arange(0, 0.5 + 0.01, 0.01)
    b_values = np.arange(-60, 10 + 0.01, 0.01)

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(loss_grid.T, origin='lower', aspect='auto',
                   cmap='magma_r',
                   extent=[w_values[0], w_values[-1],
                           b_values[0], b_values[-1]])
    ax.plot(best_w, best_b, 'w+', markersize=12, markeredgewidth=2)
    ax.set_xlabel('w (score per °C)')
    ax.set_ylabel('b (intercept)')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='MAE')
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', which='both', length=0)
    plt.tight_layout()
    plt.show()
else:
    print("\u2b1c Implement grid_search first to see the loss landscape.")

**Observe:**
- The loss landscape has a single dark valley — the region of low MAE.
- The white cross marks the best (w, b) found by grid search.
- The valley is elongated: many (w, b) combinations along a diagonal produce similar losses. A steeper slope can be compensated by a lower intercept.

In [ ]:
# Plot the best-fit line over the data
if result is not None:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(roast_temp, flavor_score, c=_ACCENT, s=20, linewidths=0)
    x_line = np.linspace(170, 240, 100)
    y_line = best_w * x_line + best_b
    ax.plot(x_line, y_line, _ACCENT, linewidth=1.2,
            label=f'score = {best_w:.4f} * temp + ({best_b:.1f})')
    ax.set_xlabel('Roast Temperature (°C)')
    ax.set_ylabel('Flavor Score (1–10)')
    ax.set_xlim(left=170)
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=9, labelcolor=_TEXT)
    tufte_axis(ax)
    plt.tight_layout()
    plt.show()
else:
    print("\u2b1c Implement grid_search first to see the best-fit line.")

### ⚡ Computational Cost

> Grid search with step size h = 0.01 over the ranges above already creates thousands of (w, b) combinations. If you halve h to 0.005, how does the total number of operations change — and what does that imply for higher-dimensional problems with three or more parameters?

<details><summary>Thought</summary>

Halving h doubles the number of grid points along each axis. With two parameters that means 2 × 2 = 4 times as many combinations. With three parameters it would be 2³ = 8 times. The cost grows exponentially with the number of parameters — this is the curse of dimensionality that motivates smarter search strategies like gradient descent.
</details>

Count the total number of arithmetic operations in a grid search. For each (w, b) combination and each data point, the model performs 4 operations: one multiply (w*x), one add (+b), one subtract (y − prediction), and one absolute value.

Useful operations: `*` for multiplication.

In [ ]:
def count_operations(n_w, n_b, n_data):
    """Return the total number of arithmetic operations in a grid search."""
    return n_w * n_b * n_data * 4


check_operations(count_operations, 81, 61, 6)

In [ ]:
# See how operations grow as h shrinks
print(f"{'h':>8}  {'n_w':>6}  {'n_b':>6}  {'operations':>12}")
print(f"{'\u2500'*8}  {'\u2500'*6}  {'\u2500'*6}  {'\u2500'*12}")
for h in [0.1, 0.05, 0.01, 0.005, 0.001]:
    n_w = len(np.arange(0, 0.5 + h, h))
    n_b = len(np.arange(-60, 10 + h, h))
    ops = count_operations(n_w, n_b, 6)
    if ops is not None:
        print(f"{h:8.3f}  {n_w:6d}  {n_b:6d}  {ops:12,}")
    else:
        print(f"{h:8.3f}  {n_w:6d}  {n_b:6d}  {'\u2b1c':>12}")

### 🏁 Recap

**What we did:**
- 🔍 Built a linear prediction model y = wx + b from scratch.
- 📏 Implemented Mean Absolute Error as our loss function.
- 🗺️ Searched the (w, b) parameter space exhaustively via grid search.
- 📊 Visualized the loss landscape and the best-fit line.
- ⚡ Counted operations and saw how cost explodes as the grid gets finer.

**Key takeaways:**
- Grid search is conceptually simple but computationally expensive.
- The cost grows with the product of grid points along every parameter axis — adding more parameters makes it impractical fast.
- This motivates smarter optimization methods like gradient descent.

**Now head back for self-check questions and key learnings in the lecture notes.**

## Take It from Here — Next Steps (Optional)

The exercises below are **optional** extensions. They deepen your intuition but are not required to follow the rest of the course. Work through them at your own pace after the session.

### 🎯 Halve the Step Size

Run grid search three times, halving h each time (0.05, 0.01, 0.005). Compare the best loss and the runtime. At what point does the improvement become negligible?

In [ ]:
import time

for h in [0.05, 0.01, 0.005]:
    t0 = time.time()
    res = grid_search(roast_temp, flavor_score, (0, 0.5), (-60, 10), h)
    elapsed = time.time() - t0
    if res is not None:
        print(f"h={h:.3f}  best_w={res[0]:.4f}  best_b={res[1]:.2f}  "
              f"MAE={res[2]:.4f}  time={elapsed:.3f}s")
    else:
        print(f"h={h:.3f}  \u2b1c not implemented")

### 🔬 Higher Dimensions

What if the model were score = w1 * temp + w2 * temp² + b (a quadratic)? A quadratic could capture the peak in flavor score around 210 °C. Now you have three parameters. Estimate how many operations a grid search with h = 0.01 over [0, 0.5] for both weights and [−60, 10] for the bias would require. Is this still feasible?

In [ ]:
# Estimate for a 3-parameter grid search
h = 0.01
n_w1 = len(np.arange(0, 0.5 + h, h))
n_w2 = len(np.arange(0, 0.5 + h, h))
n_b = len(np.arange(-60, 10 + h, h))
n_data = len(roast_temp)

# Each data point now needs 6 operations:
# w1*x, w2*x^2, w1*x + w2*x^2, +b, y - pred, |error|
ops_3d = n_w1 * n_w2 * n_b * n_data * 6
print(f"Grid points: {n_w1} x {n_w2} x {n_b} = {n_w1 * n_w2 * n_b:,}")
print(f"Total operations: {ops_3d:,}")
print(f"\nThat is {ops_3d / 1e6:.1f} million operations \u2014 still feasible,")
print(f"but adding a fourth parameter would push it into the billions.")